# Deep Learning: perche la profondita conta

Il codice del capitolo [«Deep Learning: perche la profondita conta»](https://book.paithon.it/main/DeepLearning/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q torch torchvision

## Deep Learning: perche la profondita conta

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/overview.html)


### Profondo, non solo largo


In [ ]:
from torch import nn# input: un batch di immagini RGB, shape (N, 3, 128, 128): canali primamodel = nn.Sequential(    nn.Conv2d(3, 32, 3), nn.ReLU(),    # primi strati: bordi e linee    nn.MaxPool2d(2),    nn.Conv2d(32, 64, 3), nn.ReLU(),   # strati intermedi: texture e parti    nn.MaxPool2d(2),    nn.Conv2d(64, 128, 3), nn.ReLU(),  # parti piu grandi, oggetti    nn.AdaptiveAvgPool2d(1),           # media globale di ogni feature map    nn.Flatten(),    nn.Linear(128, 10),                # la classe finale (un logit per classe))

## Reti convoluzionali (CNN)

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/reti-convoluzionali.html)


### L'architettura tipica


In [ ]:
from torch import nn# input: un batch di immagini in scala di grigi, shape (N, 1, 28, 28)model = nn.Sequential(    # blocco 1: 32 filtri 3x3, mappe grandi come l'input    nn.Conv2d(1, 32, 3, padding="same"), nn.ReLU(),    nn.MaxPool2d(2),               # 28x28 -> 14x14    # blocco 2: piu' filtri man mano che le mappe rimpiccioliscono    nn.Conv2d(32, 64, 3, padding="same"), nn.ReLU(),    nn.MaxPool2d(2),               # 14x14 -> 7x7    nn.Flatten(),                  # srotola in un vettore di 64*7*7 = 3136    nn.Linear(64 * 7 * 7, 10),     # 10 classi (logit))

## Far funzionare le reti profonde

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/ottimizzazione-regolarizzazione.html)


### Regolare il passo nel tempo


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

from torch import nn, optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)   # passo iniziale

# dimezza il learning rate quando la loss di validazione smette di scendere
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                 factor=0.5, patience=3)

for epoca in range(50):
    addestra_una_epoca(model, train_loader, criterion, optimizer)
    loss_val = valuta(model, val_loader, criterion)   # loss di validazione
    scheduler.step(loss_val)                          # decide se ridurre il passo
```


## Le architetture che hanno fatto la storia

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/architetture-storiche.html)


### Separare lo spazio dai canali: la convoluzione che sta in un telefono


In [ ]:
import torchimport torch.nn as nnC_IN, C_OUT, K, H, W = 64, 128, 3, 56, 56x = torch.randn(1, C_IN, H, W)# convoluzione ordinaria: ogni filtro guarda tutti i canali in una volta solaordinaria = nn.Conv2d(C_IN, C_OUT, K, padding=1, bias=False)# separabile: prima la parte spaziale, un filtro per canale (groups=C_IN),# poi la parte fra i canali, una 1x1 che li rimescolaseparabile = nn.Sequential(    nn.Conv2d(C_IN, C_IN, K, padding=1, groups=C_IN, bias=False),   # depthwise    nn.Conv2d(C_IN, C_OUT, 1, bias=False),                          # pointwise)def parametri(m):    return sum(p.numel() for p in m.parameters())print("stessa forma in uscita:", ordinaria(x).shape == separabile(x).shape,      tuple(separabile(x).shape))print(f"parametri, ordinaria : {parametri(ordinaria):>8,}")print(f"parametri, separabile: {parametri(separabile):>8,}")print(f"risparmio            : {parametri(ordinaria) / parametri(separabile):.2f}x")teorico = (K * K * C_OUT) / (K * K + C_OUT)print(f"previsto dalla formula: {teorico:.2f}x   (limite: {K * K}x)")

## Una rete, molti compiti: l'apprendimento multi-compito

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/multi-compito.html)


### In pratica: il guadagno si misura, e può essere negativo


In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as FD, H = 12, 64N_ETICHETTATI, N_AUSILIARI, N_TEST = 40, 800, 2000   # poche etichette dove servonodef dati(seme):    g = torch.Generator().manual_seed(seme)    n = N_AUSILIARI + N_TEST    X = torch.randn(n, D, generator=g)    w = torch.randn(D, generator=g)    nascosto = torch.tanh(X @ w)                 # la quantità che conta davvero    return X, {        "principale": nascosto,                  # etichettata solo su 40 esempi        "parente":    nascosto ** 2,             # dipende dalla STESSA quantità        "estraneo":   torch.randn(n, generator=g),    }def addestra(X, y, ausiliario, seme, passi=800):    torch.manual_seed(seme)    tronco = nn.Sequential(nn.Linear(D, H), nn.Tanh(), nn.Linear(H, H), nn.Tanh())    teste = nn.ModuleDict({k: nn.Linear(H, 1) for k in y})    ott = torch.optim.Adam(list(tronco.parameters()) + list(teste.parameters()),                           lr=3e-3)    for _ in range(passi):        # il compito principale vede 40 esempi, l'ausiliario ne vede 800        perdita = F.mse_loss(teste["principale"](tronco(X[:N_ETICHETTATI])).squeeze(-1),                             y["principale"][:N_ETICHETTATI])        if ausiliario:            perdita = perdita + F.mse_loss(                teste[ausiliario](tronco(X[:N_AUSILIARI])).squeeze(-1),                y[ausiliario][:N_AUSILIARI])        ott.zero_grad(); perdita.backward(); ott.step()    with torch.no_grad():        pred = teste["principale"](tronco(X[N_AUSILIARI:])).squeeze(-1)        return F.mse_loss(pred, y["principale"][N_AUSILIARI:]).item()base = Nonefor ausiliario in (None, "parente", "estraneo"):    errori = [addestra(*dati(s)[:2], ausiliario, s) for s in range(5)]    media = sum(errori) / len(errori)    if base is None:        base = media    nome = ausiliario or "nessuno"    print(f"ausiliario: {nome:<10} errore sul test {media:.4f}"          f"   ({100 * (media - base) / base:+.0f}%)")